# Marketing Campaign Analysis

**Project:** iFood Marketing Campaign Analysis

**Objective:** Evaluate six historical marketing campaigns, identify customer segments with stronger Campaign 6 conversion performance, and estimate the contribution of targeted outreach.

## Reproducibility
Run this notebook from top to bottom. It starts with the original CSV in `data/raw/` and regenerates the cleaned dataset, SQLite database, analysis outputs, and charts.

**Profit assumption:** Contact cost = 3 per customer; conversion revenue = 11 per conversion.


In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# Find the project root whether Jupyter starts in the root or notebooks folder
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    p for p in candidates
    if (p / "src" / "clean_data.py").exists()
)
os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())
print("Raw dataset exists:", Path("data/raw/ml_project1_data.csv").exists())


Project root: C:\Users\ragha\OneDrive\Documents\Projects\Marketing_Campaign_Analysis
Raw dataset exists: True


## 1. Data cleaning\n\nRun the cleaning pipeline from the original raw CSV.

In [2]:
%run src/clean_data.py


--- CLEANING SUMMARY ---
Original rows: 2240
Cleaned rows: 2240
Latest customer year: 2014
Missing analytical income: 25
Missing age: 3
Unique cleaned marital statuses:
Marital_Status_Clean
Married          864
Together         580
Single           480
Divorced         232
Widow             77
Other/Unusual      7

--- SAVED FILES ---
data\processed\customers_cleaned.csv
logs\cleaning_log.csv

--- FINAL VALIDATION ---
All cleaning validations passed.


## 2. SQLite database\n\nCreate the SQLite database from the cleaned customer data.

In [3]:
%run src/build_database.py

--- DATABASE CREATED ---
Database: outputs\marketing_campaigns.sqlite
Rows: 2240
Unique customer IDs: 2240
Database validation passed.


## 3. SQL analysis\n\nExecute the saved SQL queries and display their results.

In [4]:
sql_text = Path("sql/queries.sql").read_text(encoding="utf-8-sig")
queries = [q.strip() for q in sql_text.split(";") if q.strip()]

with sqlite3.connect("outputs/marketing_campaigns.sqlite") as conn:
    for i, query in enumerate(queries, start=1):
        print(f"\\n--- SQL QUERY {i} ---")
        display(pd.read_sql_query(query, conn))


\n--- SQL QUERY 1 ---


,campaign,conversions,customers_contacted,conversion_rate_pct
0,Campaign 1,144,2240,6.43
1,Campaign 2,30,2240,1.34
2,Campaign 3,163,2240,7.28
3,Campaign 4,167,2240,7.46
4,Campaign 5,163,2240,7.28
5,Campaign 6,334,2240,14.91


\n--- SQL QUERY 2 ---


,customers_contacted,conversions,cost_per_contact,revenue_per_conversion,total_contact_cost,total_conversion_revenue,campaign_profit
0,2240,334,3,11,6720,3674,-3046


\n--- SQL QUERY 3 ---


,segment,customers,conversions,conversion_rate_pct,mean_total_spend
0,PhD,486,101,20.78,672.41
1,Master,370,57,15.41,611.78
2,Graduation,1127,152,13.49,619.90
3,2n Cycle,203,22,10.84,496.53
4,Basic,54,2,3.70,81.80


\n--- SQL QUERY 4 ---


,segment,customers,conversions,conversion_rate_pct,mean_total_spend
0,Other/Unusual,7,3,42.86,571.86
1,Widow,77,19,24.68,738.82
2,Single,480,106,22.08,606.48
3,Divorced,232,48,20.69,610.63
4,Married,864,98,11.34,590.80
5,Together,580,60,10.34,608.39


## 4. SQL and Pandas verification\n\nIndependently calculate campaign conversion rates and Campaign 6 profit in Pandas, then assert that the results match SQLite.

In [5]:
%run src/verify_sql_pandas.py


--- PANDAS CAMPAIGN RESULTS ---
  campaign  conversions  customers_contacted  conversion_rate_pct
Campaign 1          144                 2240                 6.43
Campaign 2           30                 2240                 1.34
Campaign 3          163                 2240                 7.28
Campaign 4          167                 2240                 7.46
Campaign 5          163                 2240                 7.28
Campaign 6          334                 2240                14.91

--- PROFIT CROSS-CHECK ---
Customers contacted: 2240
Conversions: 334
Contact cost: 6720
Conversion revenue: 3674
Campaign 6 profit: -3046

--- VERIFICATION ---
All six campaign conversion results match SQL.
Campaign 6 profit matches SQL.
All SQL-Pandas assertions passed.


## 5. Customer segment analysis\n\nCalculate Campaign 6 conversion rates and mean spending across age, income, education, marital status, household size, tenure, and income-by-children segments. Apply the specified sample-size and chi-square criteria.

In [6]:
%run src/analyze_segments.py


--- ANALYSIS REFERENCE ---
Reference year: 2014
Reference date: 2014-06-29
Overall Campaign 6 conversion rate: 14.91 %

--- SEGMENT PERFORMANCE ---

Age Band
segment  customers  conversions  conversion_rate_pct  mean_total_spend  p_value  high_performing
  18-29        221           36                16.29            609.15 0.550448            False
    60+        316           50                15.82            749.33 0.631113            False
  30-39        557           84                15.08            519.60 0.908696            False
  40-49        676          101                14.94            554.38 0.992950            False
  50-59        467           63                13.49            684.08 0.326174            False

Income Quartile
     segment  customers  conversions  conversion_rate_pct  mean_total_spend  p_value  high_performing
Q4 (Highest)        554          149                26.90           1360.01 0.000000             True
          Q2        554           69  

## 6. Campaign comparison and confidence intervals\n\nCompare all six campaigns using Wilson 95% confidence intervals and compare average spending among converters and non-converters.

In [7]:
%run src/analyze_campaigns.py


--- CAMPAIGN RANKING AND 95% WILSON CONFIDENCE INTERVALS ---
 rank   campaign  conversions  customers  conversion_rate_pct  ci_lower_pct  ci_upper_pct
    1 Campaign 6          334       2240                14.91         13.50         16.45
    2 Campaign 4          167       2240                 7.46          6.44          8.62
    3 Campaign 3          163       2240                 7.28          6.27          8.43
    4 Campaign 5          163       2240                 7.28          6.27          8.43
    5 Campaign 1          144       2240                 6.43          5.49          7.52
    6 Campaign 2           30       2240                 1.34          0.94          1.91

--- MEAN TOTAL SPEND: CONVERTERS VS NON-CONVERTERS ---
  campaign  converter_count  mean_spend_converters  non_converter_count  mean_spend_non_converters  spend_difference
Campaign 1              144                1482.22                 2096                     545.59            936.64
Campaign 2        

## 7. Targeted profit scenario\n\nCompare the historical all-customer scenario with the mutually exclusive qualifying income-by-children groups.

In [8]:
%run src/calculate_profit.py


--- TARGETED SEGMENT DETAILS ---
income_quartile children_group  customers_contacted  historical_conversions  conversion_rate_pct  expected_conversions
   Q4 (Highest)     0 children                  386                     130                33.68                 130.0
             Q2     0 children                   51                      13                25.49                  13.0

--- PROFIT COMPARISON ---
              scenario  customers_contacted  expected_conversions  conversion_rate_pct  contact_cost  conversion_revenue  profit
      Contact Everyone                 2240                   334                14.91          6720                3674   -3046
Targeted Segments Only                  437                   143                32.72          1311                1573     262

--- ADDITIONAL METRICS ---
Break-even conversion rate: 27.27 %
Targeted conversion rate: 32.72 %
Profit improvement: 3308
Customers not contacted: 1803

--- VALIDATION ---
Profit calculations an

## 8. Generate charts\n\nGenerate the four required campaign, segment, spending, and profit charts.

In [9]:
%run src/create_charts.py

--- CHARTS CREATED ---
outputs\charts\01_campaign_conversion_ci.png - 81982 bytes
outputs\charts\02_segment_conversion_rates.png - 247533 bytes
outputs\charts\03_spend_by_conversion.png - 102946 bytes
outputs\charts\04_profit_comparison.png - 73220 bytes

All four chart files created successfully.


## Key findings

- Campaign 6 recorded 334 conversions from 2,240 customers (14.91%).
- Its 95% Wilson confidence interval is 13.50%?16.45%.
- Contacting everyone produces an estimated contribution of -3,046 under the stated assumptions.
- The selected mutually exclusive targeting groups contain 437 customers and produce an in-sample estimated contribution of +262.
- The targeted scenario requires prospective validation because the segments were selected using the same historical campaign outcomes.

## Limitations

- Segment tests are exploratory and p-values are not adjusted for multiple comparisons.
- Historical conversion rates may not persist in a future campaign.
- The profit calculation uses the dataset's contact-cost and conversion-revenue constants; it is not a complete accounting of business profitability.
